# Telecom Customer Churn Analysis - Data Exploration

This notebook focuses on the initial exploration of the telecom customer churn dataset. We'll examine the structure, quality, and basic characteristics of our data.

## Table of Contents
* [1. Setup and Imports](#setup_and_imports)
* [2. Data Loading](#data_loading)
* [3. Data Overview](#data_overview)
* [4. Data Cleaning](#data_cleaning)
    * [4.1 Column Removal](#column_removal)
    * [4.2 Dataset Splitting](#dataset_splitting)
* [5. Summary](#summary)

## Objectives
- Load and examine the dataset structure
- Understand data types and basic statistics
- Identify missing values and data quality issues
- Get initial insights into the churn patterns

In [1]:
# 1. Setup and Imports <a class="anchor" id="setup_and_imports"></a>
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

## 2. Data Loading <a class="anchor" id="data_loading"></a>

Let's start by loading our telecom customer churn dataset and taking a first look at its structure.

In [2]:
# Load the dataset
df = pd.read_csv('../dataset/telecom_customer_churn.csv')

## 3. Data Overview <a class="anchor" id="data_overview"></a>

Let's create a unified table that combines data info, statistics, and quality assessment for all features.

In [3]:
import pandas as pd
from IPython.display import display, HTML

print("COMPREHENSIVE DATA OVERVIEW")
print("=" * 80)
print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Total Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")

# Build overview data
overview_data = []
for col in df.columns:
    is_numeric = df[col].dtype in ['int64', 'float64']
    null_count = df[col].isnull().sum()
    
    row = {
        'Column': col,
        'Dtype': str(df[col].dtype),
        'Non_Null': df[col].count(),
        'Null': null_count,
        'Null_%': f"{(null_count / len(df) * 100):.2f}%"
    }
    
    if is_numeric:
        stats = df[col].describe()
        row.update({
            'Mean': f"{stats['mean']:.2f}",
            'Std': f"{stats['std']:.2f}",
            'Min': f"{stats['min']:.2f}",
            'Median': f"{stats['50%']:.2f}",
            'Max': f"{stats['max']:.2f}",
            'Unique': '',
            'Top_Values': ''
        })
    else:
        vc = df[col].value_counts()
        top = [f"{v} ({c/df[col].count()*100:.1f}%)" for v, c in vc.head(5).items()]
        row.update({
            'Mean': '', 'Std': '', 'Min': '', 'Median': '', 'Max': '',
            'Unique': df[col].nunique(),
            'Top_Values': '<br>'.join(top)
        })
    
    overview_data.append(row)

# Sort by dtype then column name
overview_df = pd.DataFrame(overview_data).sort_values(['Dtype', 'Column'])

# Create styled HTML table
html = "<table style='border-collapse: collapse; width: 100%; font-size: 12px;'>"
html += "<tr style='background-color: #333; color: white;'>"
for col in overview_df.columns:
    html += f"<th style='padding: 8px; text-align: left; border: 1px solid #ddd;'>{col}</th>"
html += "</tr>"

for _, row in overview_df.iterrows():
    html += "<tr style='border: 1px solid #ddd;'>"
    for col in overview_df.columns:
        html += f"<td style='padding: 8px; vertical-align: top; border: 1px solid #ddd;'>{row[col]}</td>"
    html += "</tr>"
html += "</table>"

display(HTML(html))

COMPREHENSIVE DATA OVERVIEW
Dataset Shape: 7043 rows × 38 columns
Total Memory Usage: 8.60 MB



Column,Dtype,Non_Null,Null,Null_%,Mean,Std,Min,Median,Max,Unique,Top_Values
Avg Monthly GB Download,float64,5517,1526,21.67%,26.19,19.59,2.00,21.00,85.00,,
Avg Monthly Long Distance Charges,float64,6361,682,9.68%,25.42,14.20,1.01,25.69,49.99,,
Latitude,float64,7043,0,0.00%,36.20,2.47,32.56,36.21,41.96,,
Longitude,float64,7043,0,0.00%,-119.76,2.15,-124.30,-119.60,-114.19,,
Monthly Charge,float64,7043,0,0.00%,63.60,31.20,-10.00,70.05,118.75,,
Total Charges,float64,7043,0,0.00%,2280.38,2266.22,18.80,1394.55,8684.80,,
Total Long Distance Charges,float64,7043,0,0.00%,749.10,846.66,0.00,401.44,3564.72,,
Total Refunds,float64,7043,0,0.00%,1.96,7.90,0.00,0.00,49.79,,
Total Revenue,float64,7043,0,0.00%,3034.38,2865.20,21.36,2108.64,11979.34,,
Age,int64,7043,0,0.00%,46.51,16.75,19.00,46.00,80.00,,


## 4. Data Cleaning <a class="anchor" id="data_cleaning"></a>

### 4.1 Column Removal <a id="column_removal"></a>

Now let's remove columns that won't be useful for our churn analysis.

In [4]:
# Drop columns that are not useful for churn analysis
potential_drops = ['Customer ID', 'Latitude', 'Longitude', 'Zip Code']
columns_to_drop = [col for col in potential_drops if col in df.columns]

# Check for similar column names if exact matches not found
if not columns_to_drop:
    for col in df.columns:
        if any(keyword in col.lower() for keyword in ['id', 'latitude', 'longitude', 'zip']):
            columns_to_drop.append(col)

if columns_to_drop:
    print(f"Dropping {len(columns_to_drop)} columns: {columns_to_drop}")
    df = df.drop(columns=columns_to_drop)
    print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
else:
    print("No specified columns found to drop")

Dropping 4 columns: ['Customer ID', 'Latitude', 'Longitude', 'Zip Code']
Dataset shape: 7,043 rows × 34 columns


### 4.2 Dataset Splitting <a id="dataset_splitting"></a>

Create separate datasets based on Customer Status for different analysis purposes.

In [5]:
# Create separate datasets based on Customer Status
import os

# Check Customer Status distribution
if 'Customer Status' in df.columns:
    print("Customer Status distribution:")
    status_counts = df['Customer Status'].value_counts()
    print(status_counts)
    
    # Create dataset with only 'Joined' customers
    df_joined = df[df['Customer Status'] == 'Joined'].copy()
    
    # Create dataset with only 'Stayed' and 'Churned' customers
    df_churn_analysis = df[df['Customer Status'].isin(['Stayed', 'Churned'])].copy()
    
    # Create dataset directory if it doesn't exist
    os.makedirs('../dataset', exist_ok=True)
    
    # Save datasets
    df_joined.to_csv('../dataset/telecom_joined_customers.csv', index=False)
    df_churn_analysis.to_csv('../dataset/telecom_churn_analysis.csv', index=False)
    
    print(f"\nDatasets created:")
    print(f"   • Joined customers: {df_joined.shape[0]:,} rows → '../dataset/telecom_joined_customers.csv'")
    print(f"   • Churn analysis: {df_churn_analysis.shape[0]:,} rows → '../dataset/telecom_churn_analysis.csv'")
    
else:
    print("'Customer Status' column not found in dataset")

Customer Status distribution:
Customer Status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64

Datasets created:
   • Joined customers: 454 rows → '../dataset/telecom_joined_customers.csv'
   • Churn analysis: 6,589 rows → '../dataset/telecom_churn_analysis.csv'


## 5. Summary <a class="anchor" id="summary"></a>

Based on our data exploration, here are the key findings and recommendations for next steps.

lorem ipsum